In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as py
import plotly.express as px


# hTERT HME1 Experiment 2

In [2]:
df = pd.read_excel("Experiment/2_hTERT_HME1/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
df

,protein_Id,description,protein_name,Peptide,MaxPepProbability,site_start,site_end,n_sites,localized_sites,phospho_sites,...,CV_raw_EGFnINS5,CV_raw_EGFnINS10,CV_raw_EGFnINS15,CV_raw_EGFnINS90,CV_raw_All,difference_EGF_vs_EGFnINS,difference_INS_vs_EGFnINS,difference_geometric_mean,min_diff,protein_total_diff
0,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGSGNFGGGR,0.9956,197,199,1,0,0,...,0.000000,0.000000,0.000000,0.000000,33.346090,0.644357,0.274540,0.420597,0.274540,4.174593
1,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGsGNFGGGR,1.0000,197,199,1,1,S199,...,1.230777,12.588830,22.938866,25.170515,50.670048,2.434020,1.123635,1.653768,1.123635,4.174593
2,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SKSEsPKEPEQLR,1.0000,2,6,1,1,S6,...,3.917276,5.877727,6.619161,2.665032,10.012151,0.255705,0.459264,0.342689,0.255705,4.174593
3,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,sKsESPK,0.9894,2,6,2,2,S2;S4,...,0.000000,0.000000,0.000000,0.000000,33.437045,0.277106,0.671679,0.431424,0.277106,4.174593
4,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,1.0000,305,316,1,0,0,...,9.209089,5.233678,18.339542,5.388291,13.267868,0.295146,0.411714,0.348591,0.295146,4.174593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50009,rev_Q5VWT5,0,0,GALMNRRGtHVNtMQIK,0.9898,303,307,2,2,T303;T307,...,0.000000,0.000000,0.000000,0.000000,9.475756,0.105879,0.085024,0.094880,0.085024,0.085024
50010,rev_Q8N6M0,0,0,LQDEIAKYMCHGDsPIQK,0.9970,135,141,1,1,S141,...,0.000000,0.000000,0.000000,0.000000,11.985616,0.139221,0.087091,0.110113,0.087091,0.087091
50011,rev_Q96M83,0,0,ERGPtIKK,0.9835,505,505,1,1,T505,...,0.000000,0.000000,0.000000,0.000000,21.585989,0.450723,0.196474,0.297582,0.196474,0.196474
50012,rev_Q9ULL1,0,0,SSASSSTSGFSVPR,0.9787,1366,1376,2,0,0,...,0.000000,0.000000,0.000000,0.000000,14.384036,0.196891,0.163787,0.179578,0.163787,0.319754


In [3]:
full_abs_list = ["abs_EGF_full_r1", "abs_EGF_full_r2", "abs_EGF_full_r3", "abs_EGF_full_r4"]
starve_abs_list = ["abs_EGF_starve_r1", "abs_EGF_starve_r2", "abs_EGF_starve_r3", "abs_EGF_starve_r4"]
EGF_abs_list = ["abs_EGF2_r1", "abs_EGF2_r2", "abs_EGF2_r3", "abs_EGF2_r4", "abs_EGF5_r1", "abs_EGF5_r2", "abs_EGF5_r3", "abs_EGF5_r4", "abs_EGF10_r1", "abs_EGF10_r2", "abs_EGF10_r3", "abs_EGF10_r4", "abs_EGF15_r1", "abs_EGF15_r2", "abs_EGF15_r3", "abs_EGF15_r4", "abs_EGF90_r1", "abs_EGF90_r2", "abs_EGF90_r3", "abs_EGF90_r4"]
INS_abs_list = ["abs_INS2_r1", "abs_INS2_r2", "abs_INS2_r3", "abs_INS2_r4", "abs_INS5_r1", "abs_INS5_r2", "abs_INS5_r3", "abs_INS5_r4", "abs_INS10_r1", "abs_INS10_r2", "abs_INS10_r3", "abs_INS10_r4", "abs_INS15_r1", "abs_INS15_r2", "abs_INS15_r3", "abs_INS15_r4", "abs_INS90_r1", "abs_INS90_r2", "abs_INS90_r3", "abs_INS90_r4"]
EGFnINS_abs_list = ["abs_EGFnINS2_r1", "abs_EGFnINS2_r2", "abs_EGFnINS2_r3", "abs_EGFnINS2_r4", "abs_EGFnINS5_r1", "abs_EGFnINS5_r2", "abs_EGFnINS5_r3", "abs_EGFnINS5_r4", "abs_EGFnINS10_r1", "abs_EGFnINS10_r2", "abs_EGFnINS10_r3", "abs_EGFnINS10_r4", "abs_EGFnINS15_r1", "abs_EGFnINS15_r2", "abs_EGFnINS15_r3", "abs_EGFnINS15_r4", "abs_EGFnINS90_r1", "abs_EGFnINS90_r2", "abs_EGFnINS90_r3", "abs_EGFnINS90_r4"]

time_points = ['Full', 'Starve', 'EGF2', 'EGF5', 'EGF10', 'EGF15', 'EGF90',
               'Full', 'Starve', 'INS2', 'INS5', 'INS10', 'INS15', 'INS90',
               'Full', 'Starve', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS15', 'EGFnINS90']

def assign_symbol(time_point):
    if "EGF" in time_point:
        if "EGFnINS" in time_point:
            return "Cross"
        else:
            return "Diamond"
        # Example symbol for "EGF"
    elif "INS" in time_point:
        return "Square"  # Example symbol for "INS"
    elif "Full" in time_point:
        return "Circle"
    elif "Starve" in time_point:
        return "Circle"



In [4]:
df_4reps = df.loc[df["n_rep"] == 4].copy()
df_for_PCA_4r = df_4reps[["site"]+full_abs_list+starve_abs_list+EGF_abs_list+INS_abs_list+EGFnINS_abs_list].copy()
EGF_data_4r = df_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGF_abs_list].copy()
INS_data_4r = df_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+INS_abs_list].copy()
EGFnINS_data_4r =df_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGFnINS_abs_list].copy()

data_all_4r = df_for_PCA_4r.set_index("site").T
data_EGF_4r = EGF_data_4r.set_index("site").T
data_INS_4r = INS_data_4r.set_index("site").T
data_EGFnINS_4r = EGFnINS_data_4r.set_index("site").T

In [5]:
df_for_imputation = df[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list + INS_abs_list + EGFnINS_abs_list].copy()

for time_point in time_points:
    replicate_cols = [col for col in df_for_imputation.columns if time_point in col]
    df_for_imputation[replicate_cols] = df_for_imputation[replicate_cols].apply(
        lambda row: row.replace(0.0, row[row != 0.0].mean()), axis=1 #I COULD USE THE MEDIAN FOR IMPUTATION
    )

imputed_data = df_for_imputation.copy()

EGF_imputed = imputed_data[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list ].copy()
INS_imputed = imputed_data[["site"]+ full_abs_list + starve_abs_list + INS_abs_list ].copy()
EGFnINS_imputed =imputed_data[["site"]+ full_abs_list + starve_abs_list + EGFnINS_abs_list].copy()

imputed_T = imputed_data.set_index("site").T
EGF_imputed_T = EGF_imputed.set_index("site").T
INS_imputed_T = INS_imputed.set_index("site").T
EGFnIND_imputed_T = EGFnINS_imputed.set_index("site").T

print(imputed_T.shape)

imputed_T = imputed_T.dropna(axis=1)
EGF_imputed_T = EGF_imputed_T.dropna(axis=1)
INS_imputed_T = INS_imputed_T.dropna(axis=1)
EGFnIND_imputed_T = EGFnIND_imputed_T.dropna(axis=1)

print(imputed_T.shape)


(68, 50014)
(68, 50011)


In [6]:
imputed_T

site,A0A2R8Y4L2_197_199_1_0~SGSGNFGGGR,A0A2R8Y4L2_197_199_1_1_S199~SGsGNFGGGR,A0A2R8Y4L2_2_6_1_1_S6~SKSEsPKEPEQLR,A0A2R8Y4L2_2_6_2_2_S2S4~sKsESPK,A0A2R8Y4L2_305_316_1_0~NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,A0A2R8Y4L2_305_316_1_1_S309~NQGGYGGSsSSSSYGSGR,A0A2R8Y4L2_305_316_1_1_S312~NQGGYGGSSSSsSYGSGR,A0A2R8Y4L2_305_316_1_1_S313~NQGGYGGSSSSSsYGSGR,A0A2R8Y4L2_305_316_1_1_S316~NQGGYGGSSSSSSYGsGR,A0A2R8Y4L2_305_316_1_1_Y314~NQGGYGGSSSSSSyGSGR,...,rev_P33076_87_116_2_2_S87T104~LLsAALSPLAEALKYAGLDtINNQSLNLTELSK,rev_P43694_328_335_1_0~AAAAAAAAALSGTTGPFSFR,rev_P57737_179_215_3_3_S190T194Y215~GTLLVLGTDPDYsPLLtSPAVDLGLVALPGGALAEAEyLLLQR,rev_P57737_697_711_1_0~ERMQNFGTSVLHEWTGMWALRSDR,rev_Q5VU65_1354_1367_3_3_Y1354T1366S1367~IEGyRFPNQVDRALVtsNGR,rev_Q5VWT5_303_307_2_2_T303T307~GALMNRRGtHVNtMQIK,rev_Q8N6M0_135_141_1_1_S141~LQDEIAKYMCHGDsPIQK,rev_Q96M83_505_505_1_1_T505~ERGPtIKK,rev_Q9ULL1_1366_1376_2_0~SSASSSTSGFSVPR,rev_Q9ULL1_866_875_2_0~LVSIMTSSSPQSNR
abs_EGF_full_r1,0.000000e+00,1.299530e+07,0.000000e+00,1.089321e+06,1.303352e+06,0.000000,0.000000,0.000000,87790.776547,13476.573983,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
abs_EGF_full_r2,5.063830e+06,0.000000e+00,7.802330e+05,0.000000e+00,1.040472e+06,0.000000,0.000000,22319.125969,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
abs_EGF_full_r3,0.000000e+00,1.306258e+07,8.016474e+05,0.000000e+00,1.418072e+06,60048.824704,0.000000,0.000000,73877.173472,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
abs_EGF_full_r4,0.000000e+00,1.553815e+07,9.353643e+05,0.000000e+00,1.547805e+06,0.000000,35650.312233,0.000000,72717.369878,0.000000,...,22136.285261,56166.237447,180525.512120,44655.893298,268592.733715,40477.275829,144071.432483,99523.486239,111663.901936,17759.500729
abs_EGF_starve_r1,0.000000e+00,2.969098e+06,0.000000e+00,2.075782e+06,1.352778e+06,0.000000,0.000000,0.000000,73508.123364,10271.311897,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
abs_EGFnINS15_r4,4.446589e+06,8.644958e+06,1.046809e+06,1.333773e+06,1.917094e+06,67816.651707,37198.835970,31449.313206,91273.546208,15438.497532,...,82136.756970,54766.548856,157335.263159,48174.680320,279344.284638,40453.319130,173038.832685,150609.393013,122881.501561,20761.550042
abs_EGFnINS90_r1,4.670694e+06,8.714140e+06,9.694752e+05,1.371579e+06,1.757922e+06,71999.840900,42490.038881,38613.791452,92014.386006,17205.356777,...,74774.221016,59563.254953,185619.360809,56806.686437,284489.593079,41940.990556,177496.159053,148947.664951,115154.539046,18539.080757
abs_EGFnINS90_r2,5.449657e+06,1.097146e+07,1.038195e+06,1.660829e+06,1.762728e+06,71999.840900,42490.038881,36666.173422,74705.207944,14940.628971,...,74774.221016,59563.254953,185619.360809,56806.686437,284489.593079,41940.990556,177496.159053,148947.664951,115154.539046,18539.080757
abs_EGFnINS90_r3,4.670694e+06,1.277781e+07,1.014930e+06,1.660829e+06,1.579118e+06,75627.650134,42490.038881,38613.791452,74624.592407,14940.628971,...,74774.221016,59563.254953,185619.360809,56806.686437,284489.593079,41940.990556,177496.159053,148947.664951,115154.539046,18539.080757


In [8]:
#PLOT with all the replicates united and separated
all_data = [EGF_imputed_T]#imputed_T, data_all_4r, EGF_imputed_T, data_EGF_4r, INS_imputed_T, data_INS_4r, EGFnIND_imputed_T, data_EGFnINS_4r]

for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       # Create a PCA dataframe for visualization
       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Visualize the PCA result
       fig = px.scatter(pca_df, x="PC1", y="PC2", color="Condition", symbol="Symbol",title="hTERT-HME1 experiment 2 PCA", width=1000, height=800)
       # fig.update_xaxes(range=[-350, 350])
       # fig.update_yaxes(range=[-200, 200])
       fig.show()

       print(data.shape)
########################################
for data in all_data:
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data)

    pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
    principal_components = pca.fit_transform(standardized_data)

    # Create a PCA dataframe for visualization
    pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
    pca_df["Time_Point"] = data.index  # Add time point info
    # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
    pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True).str.replace('^abs_', '', regex=True).str.replace('EGF_full', 'Full', regex=True).str.replace('EGF_starve', 'Starve', regex=True)
    pca_df["Symbol"] = pca_df["Condition"].apply(assign_symbol)
    # pca_df["Color"] = pca_df["Time_Point"].str.extract(r'(\d+)', expand=False)


    # Visualize the PCA result
    fig = px.scatter(
        pca_df,
        x="PC1",
        y="PC2",
        color="Condition",
        symbol="Symbol",
        title="hTERT-HME1 experiment 2 PCA",
        width=1000,
        height=800
    )

    # Increase marker size and set white background
    fig.update_traces(marker=dict(size=14, line=dict(width=1.5, color='DarkSlateGrey')))
    fig.update_layout(
        plot_bgcolor="white",
        paper_bgcolor="light grey"
        # xaxis=dict(showgrid=True, gridcolor='lightgray'),
        # yaxis=dict(showgrid=True, gridcolor='lightgray')
    )

    fig.show()

    print(data.shape)

########################################
# for data in all_data:
#        scaler = StandardScaler()
#        standardized_data = scaler.fit_transform(data)
#
#        pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
#        principal_components = pca.fit_transform(standardized_data)
#
#        # Create a PCA dataframe for visualization
#        pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
#        pca_df["Time_Point"] = data.index  # Add time point info
#        # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
#        pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)
#
#        # Visualize the PCA result
#        fig = px.scatter(pca_df, x="PC1", y="PC2", color="Time_Point", symbol="Symbol",title="PCA of Time Points and Replicates", width=1000, height=800)
#        # fig.update_xaxes(range=[-350, 350])
#        # fig.update_yaxes(range=[-200, 200])
#        fig.show()
#
#        print(data.shape)
# the outliers are : r1 for EGF2, r3 for starve, r3 for egf90

(28, 50011)


(28, 50011)


In [44]:
#Drop replicates
imputed_T_drop = imputed_T[~imputed_T.index.str.contains("r3")]

In [45]:
#PLOT HEAT MAPS
all_data = [imputed_T, data_all_4r, imputed_T_drop] # EGF_imputed_T, INS_imputed_T, EGFnIND_imputed_T]
for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=3)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2", "PC3"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True).str.replace('^abs_', '', regex=True).str.replace('EGF_full', 'Full', regex=True).str.replace('EGF_starve', 'Starve', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Calculate centroids for each condition
       centroids = pca_df.groupby("Condition")[["PC1", "PC2", "PC3"]].median() #.mean()

       # Compute pairwise distances between centroids
       dist_matrix = pd.DataFrame(
           squareform(pdist(centroids, metric='euclidean')),
           index=centroids.index,
           columns=centroids.index
       )

       if "abs_EGFnINS5_r2" in list(pca_df["Time_Point"]) and "abs_INS5_r2" in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'EGF2', 'EGF5', 'EGF10', 'EGF15', 'EGF90',
                           'INS2', 'INS5', 'INS10', 'INS15', 'INS90',
                           'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS15', 'EGFnINS90']
       elif "abs_EGFnINS5_r2" in list(pca_df["Time_Point"]) and "abs_INS5_r2" not in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS15', 'EGFnINS90']
       elif "abs_INS5_r2" in list(pca_df["Time_Point"]) and "abs_EGFnINS5_r2" not in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'INS2', 'INS5', 'INS10', 'INS15', 'INS90']
       else:
           custom_order = ['Full', 'Starve', 'EGF2', 'EGF5', 'EGF10', 'EGF15', 'EGF90']


       dist_matrix_ordered = dist_matrix.loc[custom_order, custom_order]

       text_labels = dist_matrix_ordered.round(0).astype(int).astype(str)

       fig = px.imshow(
           # dist_matrix,
           dist_matrix_ordered,
           text_auto='.0f',
           zmax=550,
           color_continuous_scale="blues",
           title="Distance Between Conditions (PCA 3D Centroids)"
       )
       fig.update_layout(width=600, height=600)
       fig.show()

       print(data.shape)

(68, 50011)


(68, 13470)


(51, 50011)


# HEK293T

In [46]:
df_HEK = pd.read_excel("Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T_diff.xlsx")
df_HEK = df_HEK.fillna(0)
df_HEK

,protein_Id,description,protein_name,Peptide,MaxPepProbability,site_start,site_end,n_sites,localized_sites,phospho_sites,...,CV_raw_EGFnINS5,CV_raw_EGFnINS10,CV_raw_EGFnINS15,CV_raw_EGFnINS90,CV_raw_All,difference_EGF_vs_EGFnINS,difference_INS_vs_EGFnINS,difference_geometric_mean,min_diff,protein_total_diff
0,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGSGNFGGGR,0.9010,197,199,1,0,0,...,0.000000,0.000000,0.000000,0.000000,34.276581,0.211143,0.080023,0.129986,0.080023,4.787742
1,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGsGNFGGGR,1.0000,197,199,1,1,S199,...,4.714165,4.992779,17.848973,14.589706,36.264607,0.627212,0.875666,0.741099,0.627212,4.787742
2,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,GGGGYGGSGDGYNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,1.0000,237,271,1,0,0,...,0.000000,0.000000,0.000000,0.000000,5.667786,0.040935,0.039200,0.040058,0.039200,4.787742
3,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,GGGGYGGSGDGyNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,1.0000,237,271,1,1,Y244,...,0.000000,0.000000,0.000000,0.000000,7.484990,0.130758,0.107351,0.118478,0.107351,4.787742
4,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,1.0000,305,316,1,0,0,...,5.630183,10.228770,17.900240,10.319898,14.330464,0.326720,0.307026,0.316720,0.307026,4.787742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36574,rev_Q96JQ0,0,0,AGAGGPGEARVtLARWPPGAEFDLPR,0.9976,1894,1894,1,1,T1894,...,0.000000,0.000000,0.000000,0.000000,18.959657,0.228274,0.150927,0.185614,0.150927,0.150927
36575,rev_Q9HB15,0,0,sRRPPPR,0.9873,419,419,1,1,S419,...,0.000000,0.000000,0.000000,0.000000,22.980237,0.385817,0.465041,0.423581,0.385817,0.385817
36576,rev_Q9UJW3,0,0,VANQLsGGHVDPITVPEMELFRsAVDLDEK,0.9969,72,89,2,2,S72;S89,...,0.000000,0.000000,0.000000,0.000000,18.165544,0.102486,0.135095,0.117666,0.102486,0.102486
36577,rev_Q9UKR3,0,0,TGACLMNDTIKGPyVQR,0.9834,73,86,1,1,Y86,...,0.000000,0.000000,0.000000,0.000000,18.677344,0.108641,0.253646,0.166001,0.108641,0.108641


In [47]:
df_HEK_4reps = df_HEK.loc[df_HEK["n_rep"] == 4].copy()
df_HEK_for_PCA_4r = df_HEK_4reps[["site"]+full_abs_list+starve_abs_list+EGF_abs_list+INS_abs_list+EGFnINS_abs_list].copy()
EGF_HEK_data_4r = df_HEK_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGF_abs_list].copy()
INS_HEK_data_4r = df_HEK_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+INS_abs_list].copy()
EGFnINS_HEK_data_4r =df_HEK_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGFnINS_abs_list].copy()

data_HEK_all_4r = df_HEK_for_PCA_4r.set_index("site").T
data_HEK_EGF_4r = EGF_HEK_data_4r.set_index("site").T
data_HEK_INS_4r = INS_HEK_data_4r.set_index("site").T
data_HEK_EGFnINS_4r = EGFnINS_HEK_data_4r.set_index("site").T

In [48]:
df_HEK_for_imputation = df_HEK[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list + INS_abs_list + EGFnINS_abs_list].copy()

for time_point in time_points:
    replicate_cols = [col for col in df_HEK_for_imputation.columns if time_point in col]
    df_HEK_for_imputation[replicate_cols] = df_HEK_for_imputation[replicate_cols].apply(
        lambda row: row.replace(0.0, row[row != 0.0].mean()), axis=1
    )

imputed_data_HEK = df_HEK_for_imputation.copy()

EGF_imputed_HEK = imputed_data_HEK[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list ].copy()
INS_imputed_HEK = imputed_data_HEK[["site"]+ full_abs_list + starve_abs_list + INS_abs_list ].copy()
EGFnINS_imputed_HEK =imputed_data_HEK[["site"]+ full_abs_list + starve_abs_list + EGFnINS_abs_list].copy()

imputed_T_HEK = imputed_data_HEK.set_index("site").T
EGF_imputed_T_HEK = EGF_imputed_HEK.set_index("site").T
INS_imputed_T_HEK = INS_imputed_HEK.set_index("site").T
EGFnIND_imputed_T_HEK = EGFnINS_imputed_HEK.set_index("site").T

print(imputed_T_HEK.shape)

imputed_T_HEK = imputed_T_HEK.dropna(axis=1)
EGF_imputed_T_HEK = EGF_imputed_T_HEK.dropna(axis=1)
INS_imputed_T_HEK = INS_imputed_T_HEK.dropna(axis=1)
EGFnIND_imputed_T_HEK = EGFnIND_imputed_T_HEK.dropna(axis=1)

print(imputed_T_HEK.shape)


(68, 36579)
(68, 36573)


In [84]:
#PLOT with all the replicates united and separated

all_data = [imputed_T_HEK]#, data_HEK_all_4r, EGF_imputed_T_HEK, data_HEK_EGF_4r, INS_imputed_T_HEK, data_HEK_INS_4r, EGFnIND_imputed_T_HEK, data_HEK_EGFnINS_4r]

for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       # Create a PCA dataframe for visualization
       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
       pca_df["Time_Point"] = data.index  # Add time point info
       # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Visualize the PCA result
       fig = px.scatter(pca_df, x="PC1", y="PC2", color="Time_Point", symbol="Symbol",title="PCA of Time Points and Replicates", width=1000, height=800)
       fig.update_xaxes(range=[-350, 350])
       fig.update_yaxes(range=[-200, 200])
       fig.show()

       print(data.shape)
for data in all_data:
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data)

    pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
    principal_components = pca.fit_transform(standardized_data)

    # Create a PCA dataframe for visualization
    pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
    pca_df["Time_Point"] = data.index  # Add time point info
    # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
    pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True).str.replace('^abs_', '', regex=True).str.replace('EGF_full', 'Full', regex=True).str.replace('EGF_starve', 'Starve', regex=True)
    pca_df["Symbol"] = pca_df["Condition"].apply(assign_symbol)
    # pca_df["Color"] = pca_df["Time_Point"].str.extract(r'(\d+)', expand=False)


    # Visualize the PCA result
    fig = px.scatter(
        pca_df,
        x="PC1",
        y="PC2",
        color="Condition",
        # color="red",
        symbol="Symbol",
        title="hTERT-HME1 experiment 1 PCA",
        width=1000,
        height=800
    )

    # Increase marker size and set white background
    fig.update_traces(marker=dict(size=14, line=dict(width=1.5, color='DarkSlateGrey')))
    fig.update_layout(
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig.show()

    print(data.shape)

# for data in all_data:
#        scaler = StandardScaler()
#        standardized_data = scaler.fit_transform(data)
#
#        pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
#        principal_components = pca.fit_transform(standardized_data)
#
#        # Create a PCA dataframe for visualization
#        pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
#        pca_df["Time_Point"] = data.index  # Add time point info
#        pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
#        pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)
#
#        # Visualize the PCA result
#        fig = px.scatter(pca_df, x="PC1", y="PC2", color="Condition", symbol="Symbol",title="PCA of Time Points and Replicates", width=1000, height=800)
#        fig.update_xaxes(range=[-350, 350])
#        fig.update_yaxes(range=[-200, 200])
#        fig.show()
#
#        print(data.shape)

(68, 36573)


(68, 36573)


In [50]:
#Drop replicates
imputed_T_HEK_drop = imputed_T_HEK[~imputed_T_HEK.index.str.contains("r1")]

In [83]:
#PLOT HEAT MAPS
all_data = [imputed_T_HEK, data_HEK_all_4r, imputed_T_HEK_drop] # [imputed_T_HEK, data_HEK_all_4r, EGF_imputed_T_HEK, data_HEK_EGF_4r, INS_imputed_T_HEK, data_HEK_INS_4r, EGFnIND_imputed_T_HEK, data_HEK_EGFnINS_4r]
for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=3)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2", "PC3"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True).str.replace('^abs_', '', regex=True).str.replace('EGF_full', 'Full', regex=True).str.replace('EGF_starve', 'Starve', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Calculate centroids for each condition
       centroids = pca_df.groupby("Condition")[["PC1", "PC2", "PC3"]].median() #.mean() .median()

       # Compute pairwise distances between centroids
       dist_matrix = pd.DataFrame(
           squareform(pdist(centroids, metric='euclidean')),
           index=centroids.index,
           columns=centroids.index
       )

       if "abs_EGFnINS5_r3" in list(pca_df["Time_Point"]) and "abs_INS5_r3" in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'EGF2', 'EGF5', 'EGF10', 'EGF15', 'EGF90',
                           'INS2', 'INS5', 'INS10', 'INS15', 'INS90',
                           'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS15', 'EGFnINS90']
       elif "abs_EGFnINS5_r3" in list(pca_df["Time_Point"]) and "abs_INS5_r3" not in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS15', 'EGFnINS90']
       elif "abs_INS5_r3" in list(pca_df["Time_Point"]) and "abs_EGFnINS5_r3" not in list(pca_df["Time_Point"]):
           custom_order = ['Full', 'Starve', 'INS2', 'INS5', 'INS10', 'INS15', 'INS90']
       else:
           custom_order = ['Full', 'Starve', 'EGF2', 'EGF5', 'EGF10', 'EGF15', 'EGF90']

       dist_matrix_ordered = dist_matrix.loc[custom_order, custom_order]

       fig = px.imshow(
           # dist_matrix,
           dist_matrix_ordered,
           text_auto='.0f',
           zmax=550,
           color_continuous_scale="blues",
           title="Distance Between Conditions (PCA 3D Centroids)"
       )
       fig.update_layout(width=600, height=600)
       fig.show()

       print(data.shape)

(68, 36573)


(68, 8493)


(51, 36573)


# hTERT HME1 Experiment 1

In [52]:
df_HME = pd.read_excel("Experiment/1_hTERT_HME1/Data/All/Renamed.xlsx", sheet_name= "raw_abundances_matrix")

In [91]:
full_abs_list = ["full_rep1", "full_rep2", "full_rep3", "full_rep4"]
starve_abs_list = ["starve_rep1", "starve_rep2", "starve_rep3", "starve_rep4"]
EGF_abs_list = ["EGF1_rep1", "EGF1_rep2", "EGF1_rep3", "EGF1_rep4", "EGF2_rep1", "EGF2_rep2", "EGF2_rep3", "EGF2_rep4", "EGF5_rep1", "EGF5_rep2", "EGF5_rep3", "EGF5_rep4", "EGF10_rep1", "EGF10_rep2", "EGF10_rep3", "EGF10_rep4", "EGF90_rep1", "EGF90_rep2", "EGF90_rep3", "EGF90_rep4"]
INS_abs_list = ["INS1_rep1", "INS1_rep2", "INS1_rep3", "INS1_rep4","INS2_rep1", "INS2_rep2", "INS2_rep3", "INS2_rep4", "INS5_rep1", "INS5_rep2", "INS5_rep3", "INS5_rep4", "INS10_rep1", "INS10_rep2", "INS10_rep3", "INS10_rep4",  "INS90_rep1", "INS90_rep2", "INS90_rep3", "INS90_rep4"]
EGFnINS_abs_list = ["EGFnINS1_rep1", "EGFnINS1_rep2", "EGFnINS1_rep3", "EGFnINS1_rep4", "EGFnINS2_rep1", "EGFnINS2_rep2", "EGFnINS2_rep3", "EGFnINS2_rep4", "EGFnINS5_rep1", "EGFnINS5_rep2", "EGFnINS5_rep3", "EGFnINS5_rep4", "EGFnINS10_rep1", "EGFnINS10_rep2", "EGFnINS10_rep3", "EGFnINS10_rep4", "EGFnINS90_rep1", "EGFnINS90_rep2", "EGFnINS90_rep3", "EGFnINS90_rep4"]

time_points = ['full', 'starve', 'EGF1', 'EGF2', 'EGF5', 'EGF10', 'EGF90',
               'full', 'starve', 'INS1', 'INS2', 'INS5', 'INS10', 'INS90',
               'full', 'starve', 'EGFnINS1', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS90']

def assign_symbol(time_point):
    if "EGF" in time_point:
        if "EGFnINS" in time_point:
            return "Cross"
        else:
            return "Diamond"
        # Example symbol for "EGF"
    elif "INS" in time_point:
        return "Square"  # Example symbol for "INS"
    elif "ull" in time_point:
        return "Circle"
    elif "tarve" in time_point:
        return "Circle"


In [54]:
df_HME_4reps = df_HME.dropna(axis=0)
df_HME_for_PCA_4r = df_HME_4reps[["site"]+full_abs_list+starve_abs_list+EGF_abs_list+INS_abs_list+EGFnINS_abs_list].copy()
EGF_HME_data_4r = df_HME_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGF_abs_list].copy()
INS_HME_data_4r = df_HME_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+INS_abs_list].copy()
EGFnINS_HME_data_4r =df_HME_for_PCA_4r[["site"]+full_abs_list+starve_abs_list+EGFnINS_abs_list].copy()

data_HME_all_4r = df_HME_for_PCA_4r.set_index("site").T
data_HME_EGF_4r = EGF_HME_data_4r.set_index("site").T
data_HME_INS_4r = INS_HME_data_4r.set_index("site").T
data_HME_EGFnINS_4r = EGFnINS_HME_data_4r.set_index("site").T

df_HME = df_HME.fillna(0)
print(df_HME_4reps.shape)
print(df_HME.shape)

(11595, 75)
(35002, 75)


In [55]:
df_HME_for_imputation = df_HME[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list + INS_abs_list + EGFnINS_abs_list].copy()

# df_HME_for_imputation = df_HME_for_imputation.head(10)
for time_point in time_points:
    replicate_cols = [col for col in df_HME_for_imputation.columns if time_point in col]
    # print(replicate_cols)
    df_HME_for_imputation[replicate_cols] = df_HME_for_imputation[replicate_cols].apply(
        lambda row: row.replace(0.0, row[row != 0.0].mean()), axis=1
    )
    # print(df_HME_for_imputation)

imputed_data_HME = df_HME_for_imputation.copy()

EGF_imputed_HME = imputed_data_HME[["site"]+ full_abs_list + starve_abs_list + EGF_abs_list ].copy()
INS_imputed_HME = imputed_data_HME[["site"]+ full_abs_list + starve_abs_list + INS_abs_list ].copy()
EGFnINS_imputed_HME =imputed_data_HME[["site"]+ full_abs_list + starve_abs_list + EGFnINS_abs_list].copy()

imputed_T_HME = imputed_data_HME.set_index("site").T
EGF_imputed_T_HME = EGF_imputed_HME.set_index("site").T
INS_imputed_T_HME = INS_imputed_HME.set_index("site").T
EGFnIND_imputed_T_HME = EGFnINS_imputed_HME.set_index("site").T

print(imputed_T_HME.shape)

imputed_T_HME = imputed_T_HME.dropna(axis=1)
EGF_imputed_T_HME = EGF_imputed_T_HME.dropna(axis=1)
INS_imputed_T_HME = INS_imputed_T_HME.dropna(axis=1)
EGFnIND_imputed_T_HME = EGFnIND_imputed_T_HME.dropna(axis=1)

print(imputed_T_HME.shape)


(68, 35002)
(68, 35001)


In [92]:
all_data = [imputed_T_HME] #,  EGF_imputed_T_HME,  INS_imputed_T_HME,  EGFnIND_imputed_T_HME] #data_HME_all_4r, data_HME_EGF_4r, data_HME_INS_4r,, data_HME_EGFnINS_4r

for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       # Create a PCA dataframe for visualization
       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_rep\d+$', '', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Visualize the PCA result
       fig = px.scatter(pca_df, x="PC1", y="PC2", color="Condition",title="PCA of Time Points and Replicates", width=1000, height=800) # symbol="Symbol"

       fig.show()

       print(data.shape)

for data in all_data:
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data)

    pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
    principal_components = pca.fit_transform(standardized_data)

    # Create a PCA dataframe for visualization
    pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
    pca_df["Time_Point"] = data.index  # Add time point info
    # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
    pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_rep\d+$', '', regex=True)
    pca_df["Symbol"] = pca_df["Condition"].apply(assign_symbol)
    # pca_df["Color"] = pca_df["Time_Point"].str.extract(r'(\d+)', expand=False)


    # Visualize the PCA result
    fig = px.scatter(
        pca_df,
        x="PC1",
        y="PC2",
        color="Condition",
        # color="red",
        symbol="Symbol",
        title="hTERT-HME1 experiment 1 PCA",
        width=1000,
        height=800
    )

    # Increase marker size and set white background
    fig.update_traces(marker=dict(size=14, line=dict(width=1.5, color='DarkSlateGrey')))
    fig.update_layout(
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig.show()

    print(data.shape)


# for data in all_data:
#        scaler = StandardScaler()
#        standardized_data = scaler.fit_transform(data)
#
#        pca = PCA(n_components=2)  # Reduce to 2 dimensions for visualization
#        principal_components = pca.fit_transform(standardized_data)
#
#        # Create a PCA dataframe for visualization
#        pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
#        pca_df["Time_Point"] = data.index  # Add time point info
#        # pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_rep\d+$', '', regex=True)
#        pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)
#
#        # Visualize the PCA result
#        fig = px.scatter(pca_df, x="PC1", y="PC2", color="Time_Point",title="PCA of Time Points and Replicates", width=1000, height=800) # symbol="Symbol"
#        fig.update_xaxes(range=[-350, 350])
#        fig.update_yaxes(range=[-200, 200])
#        fig.show()
#
#        print(data.shape)

(68, 35001)


(68, 35001)


In [57]:
#Drop replicates
imputed_T_HME_drop_r2 = imputed_T_HME[~imputed_T_HME.index.str.contains("rep2")]

In [58]:
all_data = [imputed_T_HME, data_HME_all_4r, imputed_T_HME_drop_r2]#,  #imputed_T_HME,  EGF_imputed_T_HME,  INS_imputed_T_HME,  EGFnIND_imputed_T_HME
for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=3)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2", "PC3"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_rep\d+$', '', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Calculate centroids for each condition
       centroids = pca_df.groupby("Condition")[["PC1", "PC2", "PC3"]].median() #.mean()

       # Compute pairwise distances between centroids
       dist_matrix = pd.DataFrame(
           squareform(pdist(centroids, metric='euclidean')),
           index=centroids.index,
           columns=centroids.index
       )
       if "EGFnINS5_rep3" in list(pca_df["Time_Point"]) and "INS5_rep3" in list(pca_df["Time_Point"]):
           custom_order = ['full', 'starve', 'EGF1', 'EGF2', 'EGF5', 'EGF10', 'EGF90',
                           'INS1', 'INS2', 'INS5', 'INS10', 'INS90',
                           'EGFnINS1', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS90']
       elif "EGFnINS5_rep3" in list(pca_df["Time_Point"]) and "INS5_rep3" not in list(pca_df["Time_Point"]):
           custom_order = ['full', 'starve', 'EGFnINS1', 'EGFnINS2', 'EGFnINS5', 'EGFnINS10', 'EGFnINS90']
       elif "INS5_rep3" in list(pca_df["Time_Point"]) and "EGFnINS5_rep3" not in list(pca_df["Time_Point"]):
           custom_order = ['full', 'starve', 'INS1', 'INS2', 'INS5', 'INS10', 'INS90']
       else:
           custom_order = ['full', 'starve', 'EGF1', 'EGF2', 'EGF5', 'EGF10', 'EGF90']

       dist_matrix_ordered = dist_matrix.loc[custom_order, custom_order]

       fig = px.imshow(
           # dist_matrix,
           dist_matrix_ordered,
           text_auto='.0f',
           zmax=550,
           color_continuous_scale="blues",
           title="Distance Between Conditions (PCA 3D Centroids)"
       )
       fig.update_layout(width=600, height=600)
       fig.show()

       print(data.shape)

(68, 35001)


(68, 11595)


(51, 35001)


----------------------

# END OF USEFULL CODE

----------------

In [59]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# List of your datasets
all_data = [
    imputed_T_HEK, data_HEK_all_4r,
    EGF_imputed_T_HEK, data_HEK_EGF_4r,
    INS_imputed_T_HEK, data_HEK_INS_4r,
    EGFnIND_imputed_T_HEK, data_HEK_EGFnINS_4r
]
titles = [
    "T_HEK Imputed", "T_HEK Raw",
    "EGF Imputed", "EGF Raw",
    "INS Imputed", "INS Raw",
    "EGFnINS Imputed", "EGFnINS Raw"
]

# Create subplot grid (2 columns)
rows = (len(all_data) + 1) // 2
fig = make_subplots(rows=rows, cols=2, subplot_titles=titles)

# Loop over each dataset and subplot location
for idx, data in enumerate(all_data):
    row = idx // 2 + 1
    col = idx % 2 + 1

    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data)

    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(standardized_data)

    pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
    pca_df["Time_Point"] = data.index
    pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
    pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

    # Add a scatter trace for each condition group
    for condition in pca_df["Condition"].unique():
        cond_df = pca_df[pca_df["Condition"] == condition]
        fig.add_trace(
            go.Scatter(
                x=cond_df["PC1"],
                y=cond_df["PC2"],
                mode='markers',
                name=condition,
                legendgroup=condition,
                showlegend=(idx == 0),  # Only show legend in first plot
                marker=dict(symbol="circle"),
                text=cond_df["Time_Point"]
            ),
            row=row,
            col=col
        )

# Set layout
fig.update_layout(
    height=rows * 400,
    width=1000,
    title_text="PCA Plots in 2 Columns",
    showlegend=True
)

fig.update_xaxes(range=[-350, 350])
fig.update_yaxes(range=[-200, 200])
fig.show()


In [60]:
all_data = [EGFnIND_imputed_T]
for data in all_data:
       scaler = StandardScaler()
       standardized_data = scaler.fit_transform(data)

       pca = PCA(n_components=3)  # Reduce to 2 dimensions for visualization
       principal_components = pca.fit_transform(standardized_data)

       pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2", "PC3"])
       pca_df["Time_Point"] = data.index  # Add time point info
       pca_df["Condition"] = pca_df["Time_Point"].str.replace(r'_r\d+$', '', regex=True)
       pca_df["Symbol"] = pca_df["Time_Point"].apply(assign_symbol)

       # Visualize the PCA result
       fig = px.scatter_3d(pca_df, x="PC1", y="PC2", z="PC3", color="Condition", symbol="Symbol",title="PCA of Time Points and Replicates", width=500, height=400)
       fig.update_xaxes(range=[-350, 350])
       fig.update_yaxes(range=[-200, 200])
       fig.show()
       fig.update_layout(width=900, height=700)
       # fig.write_html("3333.html")

       print(data.shape)

(28, 50014)
